# 💰 LoanGenie — Advanced Agentic AI (Part 2)
### Multi-Agent · Streaming · Guardrails & PII · Observability

---

This notebook is the **advanced continuation** of the LoanGenie project. It assumes you have
already completed the console setup and the core modules (1–10) in Part 1:

| Resource | Expected |
|----------|----------|
| Bedrock model access | Amazon Nova Pro enabled |
| DynamoDB table | `LoanApplicants` |
| AgentCore memory | `loan_applicants` namespace |
| Knowledge Base | `LoanProductsKB` (synced) |
| SageMaker role | 8 policies attached |

**What you'll add here — production-grade capabilities:**

| Module | Capability | Why it matters |
|--------|-----------|----------------|
| A | Multi-agent orchestration | Split eligibility, advisory & docs into specialist agents |
| B | Streaming + async tools | Real-time token streaming, parallel tool calls |
| C | Guardrails + PII redaction | Mask Aadhaar/PAN, block toxic content — compliance |
| D | Observability | CloudWatch metrics, tracing, per-query cost tracking |

> **Region:** `us-east-1` · **Model:** Amazon Nova Pro

In [1]:
# [CONFIG] — resource IDs + ALL imports used anywhere in this notebook
# (centralized here so re-running any single cell later never hits a missing-import NameError)
import boto3, json, os, time, re, uuid, asyncio, subprocess, zipfile, shutil
import urllib.request, urllib.error
from datetime import datetime, timezone

REGION     = 'us-east-1'
MODEL_ID   = 'us.amazon.nova-pro-v1:0'
ACCOUNT_ID = boto3.client('sts').get_caller_identity()['Account']

KB_ID            = 'HGLSKTH8OX'         # your LoanProductsKB ID
AGENTCORE_MEM_ID = 'application_loan-Ro4cqL8XUg'                    # paste your loan_applicants memory ID
APPLICANT_TABLE  = 'LoanApplicants'

print(f'Account : {ACCOUNT_ID}')
print(f'KB_ID   : {KB_ID}')
print('✅ Config + imports loaded')


Account : 831963379350
KB_ID   : HGLSKTH8OX
✅ Config + imports loaded


In [2]:
# Install dependencies (if not already in this kernel)
!pip install strands-agents strands-agents-tools boto3 -q
print('✅ Ready')

✅ Ready


In [3]:
# Markdown rendering helper — use show() instead of print() for formatted output
from IPython.display import Markdown, display

def show(response, title=None):
    """Render an agent response as clean formatted markdown in the notebook."""
    text = str(response)
    if title:
        text = f'### {title}\n\n' + text
    display(Markdown(text))

print('✅ show() markdown renderer ready — use it instead of print() for agent output')


✅ show() markdown renderer ready — use it instead of print() for agent output


---
## Shared Financial Tools
These are reused by every specialist agent below.

In [4]:
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def check_eligibility(monthly_income: float, credit_score: float, loan_amount: float) -> str:
    """Check loan eligibility from income, credit score, and requested amount."""
    max_loan = monthly_income * 60
    reasons, eligible = [], True
    if credit_score < 650: eligible=False; reasons.append('Credit score below 650')
    if loan_amount > max_loan: eligible=False; reasons.append('Exceeds max eligible')
    if monthly_income < 25000: eligible=False; reasons.append('Income below 25000')
    band = '10.5%-12.5%' if credit_score >= 750 else '12.5%-15%'
    return json.dumps({'eligible':eligible,'max_eligible_amount':int(max_loan),
                       'reasons':reasons or ['Meets all criteria'],'interest_rate_band':band})

@tool
def calculate_emi(principal: float, annual_rate: float, tenure_months: int) -> str:
    """Calculate monthly EMI, total payable, and total interest."""
    r = annual_rate/12/100
    emi = principal/tenure_months if r==0 else principal*r*(1+r)**tenure_months/((1+r)**tenure_months-1)
    return json.dumps({'monthly_emi':round(emi,2),'total_payable':round(emi*tenure_months,2),
                       'total_interest':round(emi*tenure_months-principal,2)})

@tool
def search_loan_products(query: str) -> str:
    """Search loan product terms, rates, documents, and FAQs from the knowledge base."""
    if not KB_ID:
        return 'Knowledge base not configured.'
    client = boto3.client('bedrock-agent-runtime', region_name=REGION)
    resp = client.retrieve(
        knowledgeBaseId=KB_ID,
        retrievalQuery={'text': query},
        retrievalConfiguration={
            'managedSearchConfiguration': {}   # ← correct for managed KB
        }
    )
    results = resp.get('retrievalResults', [])
    if not results:
        return 'No relevant information found.'
    return '\n\n'.join(
        f'[{r.get("location",{}).get("s3Location",{}).get("uri","doc").split("/")[-1]}] {r["content"]["text"]}'
        for r in results
    )

---
# MODULE A — Multi-Agent Orchestration
### A supervisor routes each query to the right specialist

In production, a single agent juggling every task becomes hard to tune and debug. The
supervisor pattern splits responsibility:

```
                    ┌──────────────────┐
   User query  ───▶ │  Supervisor Agent │  (routes by intent)
                    └────────┬─────────┘
              ┌──────────────┼──────────────┐
              ▼              ▼              ▼
      ┌─────────────┐ ┌─────────────┐ ┌─────────────┐
      │ Eligibility │ │  Advisory   │ │ Documents   │
      │ Specialist  │ │ Specialist  │ │ Specialist  │
      └─────────────┘ └─────────────┘ └─────────────┘
```

Each specialist has a focused prompt and only the tools it needs — easier to test,
cheaper to run, and safer for compliance.

In [5]:
# Define the three specialist agents as callable tools for the supervisor

@tool
def eligibility_specialist(query: str) -> str:
    """Handle loan eligibility questions."""
    agent = Agent(
        model=BedrockModel(model_id=MODEL_ID),
        system_prompt='''You are an eligibility specialist for an Indian lending company.
Use check_eligibility to assess applications.
Format in markdown with **bold** for key figures.
No reasoning steps in output. Final answer only.''',
        tools=[check_eligibility]
    )
    return str(agent(query))

@tool
def advisory_specialist(query: str) -> str:
    """Handle EMI calculations and loan advice."""
    agent = Agent(
        model=BedrockModel(model_id=MODEL_ID),
        system_prompt='''You are a loan advisory specialist for an Indian lending company.
Use calculate_emi for repayment figures.
Use search_loan_products for rates and terms.
Format in markdown with **bold** for key figures.
No reasoning steps in output. Final answer only.''',
        tools=[calculate_emi, search_loan_products]
    )
    return str(agent(query))

@tool
def documents_specialist(query: str) -> str:
    """Handle document and process questions."""
    agent = Agent(
        model=BedrockModel(model_id=MODEL_ID),
        system_prompt='''You are a documentation specialist for an Indian lending company.
ALWAYS call search_loan_products first before answering.
Only state documents returned by the knowledge base — PAN, Aadhaar, ITR, salary slips etc.
Never use generic documents like driver license or passport.
Format in markdown with bullet points.
No reasoning steps in output. Final answer only.''',
        tools=[search_loan_products]
    )
    return str(agent(query))

print('✅ Three specialist agents defined')

✅ Three specialist agents defined


In [6]:
# The supervisor agent — routes to specialists
supervisor = Agent(
    model=BedrockModel(model_id=MODEL_ID),
    system_prompt='''You are the LoanGenie supervisor. Route each customer query to the
right specialist tool:
- eligibility_specialist: eligibility, income/credit assessment
- advisory_specialist: EMI calculations, rates, repayment advice
- documents_specialist: required documents, processing times

For multi-part questions, call multiple specialists and combine their answers.
Present a single, coherent response to the customer.

Format your answer in clean markdown:
- Use ## headers for each section (Eligibility, EMI, Documents)
- Use **bold** for key figures like amounts and rates
- Use bullet points for lists
- No internal reasoning or thinking steps in the output''',
    tools=[eligibility_specialist, advisory_specialist, documents_specialist]
)

# Test routing — a multi-part query should hit multiple specialists
response = supervisor(
    'I earn 85000 a month with a credit score of 740. Am I eligible for a 500000 loan, '
    'what would the EMI be over 36 months at 13%, and what documents will I need?'
)
show(response)

<thinking> The user's query includes questions about loan eligibility, EMI calculations, and required documents. Therefore,

 I need to call all three specialists: eligibility_specialist, advisory_specialist, and documents_specialist

. </thinking>

Tool #1: eligibility_specialist

Tool #2: advisory_specialist

Tool #3: documents_specialist


<thinking> To determine if the user is eligible for the loan, I need to use the check_eligibility

<thinking> I need to determine the required documents for a 500000 loan by searching the

...

 EMI Calculation
- **Loan Amount:** ₹5,00,000
- **Interest Rate:** 13%
- **Tenure:** 36 months
- **Monthly

 EMI:** **₹16,846.98**
- **Total Payable:** **₹606,491.14**
- **Total Interest:** **₹

106,491.14**

## Required Documents
- PAN card
- Aadhaar card
- Income Tax Returns (ITR) for the last two years
- Salary slips

 for the last six months
- Bank statements for the last six months
- Proof of address (utility bill or rental agreement)

## Loan Eligibility
- **Eligibility:** You are **eligible** for a loan.
- **Maximum Eligible Amount:** ₹51,00,000
- **Reasons:** Meets all criteria
- **Interest Rate Band:** **12.5%-15%**

## EMI Calculation
- **Loan Amount:** ₹5,00,000
- **Interest Rate:** 13%
- **Tenure:** 36 months
- **Monthly EMI:** **₹16,846.98**
- **Total Payable:** **₹606,491.14**
- **Total Interest:** **₹106,491.14**

## Required Documents
- PAN card
- Aadhaar card
- Income Tax Returns (ITR) for the last two years
- Salary slips for the last six months
- Bank statements for the last six months
- Proof of address (utility bill or rental agreement)


In [7]:
# Test individual routing — confirm the supervisor picks the right specialist
tests = [
    'Am I eligible for 400000 with income 70000 and score 690?',   # -> eligibility
    'What is the EMI on 600000 at 12% for 48 months?',              # -> advisory
    'What documents does a salaried applicant need?',               # -> documents
]
for t in tests:
    print(f'\n❓ {t}')
    show(supervisor(t))
    print('-'*60)


❓ Am I eligible for 400000 with income 70000 and score 690?


<thinking> The user's new query is about loan eligibility with different income and credit score. I need to

 call the eligibility_specialist to assess this. </thinking> 
Tool #4: eligibility_specialist


<thinking> The user has provided all the required parameters to check eligibility: monthly income, credit score, and loan

 amount. I need to use the `check_eligibility` tool to assess the loan application.

...

 Appointment Order
- Proof of Address (Utility Bill, Rental Agreement, etc.)

## Required Documents for Salaried Applicant
- PAN Card
- Aadhaar Card
- Income Tax Returns (I

TR) for the last two years
- Salary Slips for the last three months
- Bank Statements for the last six months
- Employment Letter or Appointment Order
- Proof of Address (Utility Bill, Rental Agreement, etc.)

## Required Documents for Salaried Applicant
- PAN Card
- Aadhaar Card
- Income Tax Returns (ITR) for the last two years
- Salary Slips for the last three months
- Bank Statements for the last six months
- Employment Letter or Appointment Order
- Proof of Address (Utility Bill, Rental Agreement, etc.)


------------------------------------------------------------


---
# MODULE B — Streaming Responses + Async Tool Calls
### Stream tokens as they generate; run independent tools in parallel

**Why streaming?** A loan advisory answer can take several seconds. Streaming shows the
customer text as it's generated instead of a frozen spinner — critical for chat UX.

**Why async tools?** When a query needs both an EMI calculation and a product lookup,
running them concurrently cuts latency roughly in half.

In [8]:
# Streaming responses — token-by-token output using Strands async iterator
import asyncio

async def stream_response(agent, query):
    """Stream the agent's response token by token."""
    print('🤖 ', end='', flush=True)
    async for event in agent.stream_async(query):
        if 'data' in event:
            print(event['data'], end='', flush=True)
    print()

stream_agent = Agent(
    model=BedrockModel(model_id=MODEL_ID),
    system_prompt='You are a loan advisor. Be concise and helpful.',
    tools=[check_eligibility, calculate_emi, search_loan_products]
)

# Run the streaming demo
await stream_response(stream_agent, 'Explain how personal loan interest rates are decided.')

🤖 

<thinking<thinking

>>

 To To

 explain explain

...

 a a

 more more

 targeted targeted

 question question

..

In [9]:
# Direct Bedrock streaming via converse_stream — lower-level control
bedrock = boto3.client('bedrock-runtime', region_name=REGION)

def stream_bedrock(prompt):
    """Stream directly from Bedrock's converse_stream API."""
    resp = bedrock.converse_stream(
        modelId=MODEL_ID,
        messages=[{'role':'user','content':[{'text':prompt}]}],
        inferenceConfig={'maxTokens':300,'temperature':0.3}
    )
    print('🤖 ', end='', flush=True)
    for event in resp['stream']:
        if 'contentBlockDelta' in event:
            delta = event['contentBlockDelta']['delta'].get('text', '')
            print(delta, end='', flush=True)
    print()

stream_bedrock('In one paragraph, explain what a credit score means for loan eligibility.')

🤖 

A

 credit score is a numerical

 representation of an

 individual's creditworthiness,

...

 better

 loan terms and improving overall

 financial

 health

.

In [10]:
# Async parallel tool calls — run independent lookups concurrently
import asyncio, time

async def async_eligibility(income, score, amount):
    return json.loads(check_eligibility(income, score, amount))

async def async_emi(principal, rate, months):
    return json.loads(calculate_emi(principal, rate, months))

async def parallel_loan_analysis(income, score, amount, rate, months):
    """Run eligibility and EMI calculations concurrently."""
    start = time.time()
    eligibility, emi = await asyncio.gather(
        async_eligibility(income, score, amount),
        async_emi(amount, rate, months)
    )
    elapsed = (time.time() - start) * 1000
    return {'eligibility': eligibility, 'emi': emi, 'elapsed_ms': round(elapsed, 1)}

result = await parallel_loan_analysis(85000, 740, 500000, 13, 36)
print(json.dumps(result, indent=2))
print(f'\n✅ Both tools ran concurrently in {result["elapsed_ms"]}ms')

{
  "eligibility": {
    "eligible": true,
    "max_eligible_amount": 5100000,
    "reasons": [
      "Meets all criteria"
    ],
    "interest_rate_band": "12.5%-15%"
  },
  "emi": {
    "monthly_emi": 16846.98,
    "total_payable": 606491.14,
    "total_interest": 106491.14
  },
  "elapsed_ms": 0.2
}

✅ Both tools ran concurrently in 0.2ms


---
# MODULE C — Guardrails + PII Redaction (Compliance)
### Mask Aadhaar/PAN, block toxic content, deny out-of-scope topics

Lending is heavily regulated. Two layers of protection:

1. **PII redaction** — customers paste Aadhaar/PAN numbers; these must never be logged
   or sent to the model in the clear.
2. **Bedrock Guardrails** — block toxic content and topics the agent shouldn't discuss
   (e.g. tax evasion, guaranteed-approval promises).

In [11]:
# PII redaction — mask Aadhaar, PAN, phone, and email before processing

def redact_pii(text):
    """Redact Indian PII patterns before the text reaches the model or logs."""
    patterns = {
        'AADHAAR': (r'\b\d{4}\s?\d{4}\s?\d{4}\b', '[AADHAAR_REDACTED]'),
        'PAN':     (r'\b[A-Z]{5}\d{4}[A-Z]\b', '[PAN_REDACTED]'),
        'PHONE':   (r'\b[6-9]\d{9}\b', '[PHONE_REDACTED]'),
        'EMAIL':   (r'\b[\w.-]+@[\w.-]+\.\w+\b', '[EMAIL_REDACTED]'),
    }
    found = []
    for label, (pattern, replacement) in patterns.items():
        if re.search(pattern, text):
            found.append(label)
            text = re.sub(pattern, replacement, text)
    return text, found


# Test PII redaction
test_input = ('My PAN is ABCDE1234F, Aadhaar 1234 5678 9012, '
              'phone 9876543210, email rahul@example.com. Am I eligible for a loan?')
clean, found = redact_pii(test_input)
show(test_input, title='Original')
show(clean, title='Redacted')
print(f'✅ PII types redacted: {found}')


### Original

My PAN is ABCDE1234F, Aadhaar 1234 5678 9012, phone 9876543210, email rahul@example.com. Am I eligible for a loan?

### Redacted

My PAN is [PAN_REDACTED], Aadhaar [AADHAAR_REDACTED], phone [PHONE_REDACTED], email [EMAIL_REDACTED]. Am I eligible for a loan?

✅ PII types redacted: ['AADHAAR', 'PAN', 'PHONE', 'EMAIL']


In [12]:
# Wrap the agent so every input is redacted before it reaches the model
def safe_agent_call(agent, user_input):
    """Redact PII, log the safe version, then call the agent."""
    clean_input, pii_found = redact_pii(user_input)
    if pii_found:
        print(f'⚠️  Redacted PII before processing: {pii_found}')
    response = agent(clean_input)
    return str(response)


safe_agent = Agent(
    model=BedrockModel(model_id=MODEL_ID),
    system_prompt='You are a loan advisor. Use the tools. Never ask for or store PII like Aadhaar or PAN.',
    tools=[check_eligibility, calculate_emi]
)

result = safe_agent_call(
    safe_agent,
    'My PAN is ABCDE1234F. I earn 90000 with score 760. Eligible for 600000?'
)
print('\nAgent response:')
show(result)

⚠️  Redacted PII before processing: ['PAN']


<thinking> To determine the user's eligibility for a loan of 600,000,

 I need to use the `check_eligibility` tool. The user has provided their monthly income and credit score. However, the user has also provided their PAN, which is Personally Identifiable

 Information (PII) that should not be stored or asked for. I will proceed with the eligibility

 check using only the provided income, credit score, and requested loan amount. </thinking>

Tool #1: check_eligibility


...

<thinking> The user is eligible for the loan, and I have calculated the EMI, total payable amount, and

 total interest for a loan of 600,000. I will now present this information to the user. </thinking>

You are eligible for a loan of 600,

000. Here are the details:
- Monthly EMI: 10,031.80
- Total Payable: 601,908.20


- Total Interest: 19,08.20

If you need further assistance or have any questions, feel free to ask!
Agent response:


<thinking> The user is eligible for the loan, and I have calculated the EMI, total payable amount, and total interest for a loan of 600,000. I will now present this information to the user. </thinking>

You are eligible for a loan of 600,000. Here are the details:
- Monthly EMI: 10,031.80
- Total Payable: 601,908.20
- Total Interest: 19,08.20

If you need further assistance or have any questions, feel free to ask!


In [13]:
# Bedrock Guardrails — create a guardrail that blocks denied topics and toxic content
bedrock_client = boto3.client('bedrock', region_name=REGION)
GUARDRAIL_ID = None
GUARDRAIL_VERSION = 'DRAFT'

try:
    guardrail = bedrock_client.create_guardrail(
        name='loangenie-guardrail',
        description='Blocks guaranteed-approval promises, tax evasion, and toxic content',
        topicPolicyConfig={
            'topicsConfig': [
                {
                    'name': 'GuaranteedApproval',
                    'definition': 'Promises or guarantees of loan approval without verification',
                    'examples': ['You are guaranteed approval', 'I promise your loan will be approved'],
                    'type': 'DENY'
                },
                {
                    'name': 'TaxEvasion',
                    'definition': 'Advice on evading taxes or hiding income',
                    'examples': ['How do I hide income from the bank', 'How to evade tax on my loan'],
                    'type': 'DENY'
                }
            ]
        },
        contentPolicyConfig={
            'filtersConfig': [
                {'type': 'HATE', 'inputStrength': 'HIGH', 'outputStrength': 'HIGH'},
                {'type': 'INSULTS', 'inputStrength': 'HIGH', 'outputStrength': 'HIGH'},
            ]
        },
        blockedInputMessaging='I can only help with legitimate loan eligibility and advisory questions.',
        blockedOutputsMessaging='I cannot provide that information.'
    )
    GUARDRAIL_ID = guardrail['guardrailId']
    print(f'✅ Guardrail created: {GUARDRAIL_ID}')
    ver = bedrock_client.create_guardrail_version(guardrailIdentifier=GUARDRAIL_ID)
    GUARDRAIL_VERSION = ver['version']
    print(f'✅ Guardrail version: {GUARDRAIL_VERSION}')
except bedrock_client.exceptions.ConflictException:
    # Already exists — look it up instead of failing
    existing = bedrock_client.list_guardrails()['guardrails']
    match = next((g for g in existing if g['name'] == 'loangenie-guardrail'), None)
    if match:
        GUARDRAIL_ID = match['id']
        print(f'ℹ️  Guardrail already exists, reusing: {GUARDRAIL_ID}')
except Exception as e:
    print(f'⚠️  Could not create guardrail: {e}')
    print('   If this is AccessDenied, your SageMaker execution role needs bedrock:CreateGuardrail.')
    print('   The rest of the notebook will still run — guarded_call() just won\'t be testable below.')


ℹ️  Guardrail already exists, reusing: ygqbclvqr9wj


In [14]:
# Apply the guardrail to a Bedrock call
def guarded_call(prompt, guardrail_id, guardrail_version='DRAFT'):
    """Call Bedrock with a guardrail applied."""
    bedrock_rt = boto3.client('bedrock-runtime', region_name=REGION)
    try:
        resp = bedrock_rt.converse(
            modelId=MODEL_ID,
            messages=[{'role':'user','content':[{'text':prompt}]}],
            guardrailConfig={'guardrailIdentifier':guardrail_id,'guardrailVersion':guardrail_version}
        )
        return resp['output']['message']['content'][0]['text']
    except Exception as e:
        return f'Guardrail response: {e}'

# Test — a denied topic should be blocked
if GUARDRAIL_ID:
    print(guarded_call('Can you guarantee my loan will be approved?', GUARDRAIL_ID, GUARDRAIL_VERSION))
    print('✅ Guarded call pattern ready')
else:
    print('⏭️  Skipped — no guardrail was created above (see warning in the previous cell).')


I can only help with legitimate loan eligibility and advisory questions.
✅ Guarded call pattern ready


---
# MODULE D — Observability
### CloudWatch metrics, request tracing, and per-query cost tracking

You can't run an agent in production without knowing: how many queries, how fast, how
much they cost, and where they fail. This module adds a lightweight observability layer.

In [15]:
# Per-query cost + latency tracking

# Nova Pro pricing (per 1M tokens) — verify current rates at aws.amazon.com/bedrock/pricing
NOVA_PRO_INPUT_PER_1M  = 0.80
NOVA_PRO_OUTPUT_PER_1M = 3.20

class LoanGenieMetrics:
    """Tracks latency, token usage, and cost per query."""
    def __init__(self):
        self.queries = []

    def track(self, query, response, input_tokens, output_tokens, latency_ms):
        cost = (input_tokens/1e6*NOVA_PRO_INPUT_PER_1M +
                output_tokens/1e6*NOVA_PRO_OUTPUT_PER_1M)
        record = {
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'query': query[:50],
            'input_tokens': input_tokens,
            'output_tokens': output_tokens,
            'latency_ms': round(latency_ms, 1),
            'cost_usd': round(cost, 6),
        }
        self.queries.append(record)
        return record

    def summary(self):
        if not self.queries: return {'total_queries': 0, 'total_cost_usd': 0, 'avg_latency_ms': 0, 'total_tokens': 0}
        total_cost = sum(q['cost_usd'] for q in self.queries)
        avg_latency = sum(q['latency_ms'] for q in self.queries)/len(self.queries)
        total_tokens = sum(q['input_tokens']+q['output_tokens'] for q in self.queries)
        return {
            'total_queries': len(self.queries),
            'total_cost_usd': round(total_cost, 6),
            'avg_latency_ms': round(avg_latency, 1),
            'total_tokens': total_tokens,
            'projected_cost_per_1000_queries': round(total_cost/len(self.queries)*1000, 2)
        }

metrics = LoanGenieMetrics()
print('✅ Metrics tracker ready')


✅ Metrics tracker ready


In [16]:
# Instrumented agent call — capture tokens, latency, cost via Bedrock converse
bedrock = boto3.client('bedrock-runtime', region_name=REGION)

def tracked_query(prompt):
    """Run a query and capture full observability metrics."""
    start = time.time()
    resp = bedrock.converse(
        modelId=MODEL_ID,
        messages=[{'role':'user','content':[{'text':prompt}]}],
        inferenceConfig={'maxTokens':400,'temperature':0.3}
    )
    latency_ms = (time.time() - start) * 1000

    usage = resp['usage']
    answer = resp['output']['message']['content'][0]['text']
    record = metrics.track(prompt, answer,
        usage['inputTokens'], usage['outputTokens'], latency_ms)

    print(f'📊 {record["latency_ms"]}ms | '
          f'{record["input_tokens"]}+{record["output_tokens"]} tokens | '
          f'${record["cost_usd"]:.6f}')
    return answer

# Run several tracked queries
for q in [
    'What is the minimum credit score for a personal loan?',
    'Explain how EMI is calculated.',
    'What documents does a self-employed applicant need?',
]:
    tracked_query(q)

print('\n=== SESSION SUMMARY ===')
print(json.dumps(metrics.summary(), indent=2))

📊 2547.5ms | 11+400 tokens | $0.001289


📊 2217.5ms | 6+400 tokens | $0.001285


📊 2185.7ms | 10+400 tokens | $0.001288

=== SESSION SUMMARY ===
{
  "total_queries": 3,
  "total_cost_usd": 0.003862,
  "avg_latency_ms": 2316.9,
  "total_tokens": 1227,
  "projected_cost_per_1000_queries": 1.29
}


In [17]:
# Publish custom metrics to CloudWatch
cloudwatch = boto3.client('cloudwatch', region_name=REGION)

def publish_metrics(metrics_obj):
    """Push LoanGenie metrics to CloudWatch for dashboards and alarms."""
    summary = metrics_obj.summary()
    if not summary.get('total_queries'):
        print('No metrics to publish'); return

    try:
        cloudwatch.put_metric_data(
            Namespace='LoanGenie',
            MetricData=[
                {'MetricName':'QueryCount','Value':summary['total_queries'],'Unit':'Count'},
                {'MetricName':'AvgLatency','Value':summary['avg_latency_ms'],'Unit':'Milliseconds'},
                {'MetricName':'TotalCost','Value':summary['total_cost_usd'],'Unit':'None'},
                {'MetricName':'TotalTokens','Value':summary['total_tokens'],'Unit':'Count'},
            ]
        )
        print('✅ Metrics published to CloudWatch namespace: LoanGenie')
        print('   View at: CloudWatch → Metrics → LoanGenie')
    except Exception as e:
        print(f'⚠️  Could not publish to CloudWatch: {e}')
        print('   If AccessDenied, your SageMaker execution role needs cloudwatch:PutMetricData.')

publish_metrics(metrics)


✅ Metrics published to CloudWatch namespace: LoanGenie
   View at: CloudWatch → Metrics → LoanGenie


In [18]:
# Simple request tracing — log the full lifecycle of a query

def traced_query(agent, user_query):
    """Trace a query through redaction → agent → response with a trace ID."""
    trace_id = str(uuid.uuid4())[:8]
    trace = {'trace_id': trace_id, 'steps': []}

    def log(step, detail):
        trace['steps'].append({'step': step, 'detail': detail,
                               'ts': datetime.now(timezone.utc).strftime('%H:%M:%S.%f')[:-3]})

    log('received', f'query length {len(user_query)}')
    clean, pii = redact_pii(user_query)
    log('pii_redaction', f'redacted: {pii}' if pii else 'no PII found')

    start = time.time()
    response = str(agent(clean))
    log('agent_response', f'{(time.time()-start)*1000:.0f}ms, {len(response)} chars')

    print(f'🔍 TRACE {trace_id}')
    for s in trace['steps']:
        print(f'   [{s["ts"]}] {s["step"]}: {s["detail"]}')
    return response, trace


demo_agent = Agent(
    model=BedrockModel(model_id=MODEL_ID),
    system_prompt='You are a loan advisor.',
    tools=[check_eligibility, calculate_emi]
)
response, trace = traced_query(demo_agent,
    'My PAN is ABCDE1234F. Am I eligible for 500000 with income 80000, score 720?')
print(f'\n{response}')


<thinking> To determine the user's eligibility for a loan, I need to use the `check_elig

ibility` tool with the provided income, credit score, and requested loan amount. </thinking>

Tool #1: check_eligibility


<thinking> The user is eligible for a loan, but the maximum eligible amount is 48000

00, not 5000000. I should inform the user of their eligibility status and the maximum eligible amount. Additionally, I should provide the interest rate band for the loan

. </thinking>

You are eligible for a loan, but the maximum eligible amount is 4800000, not 5000000. The interest rate band

 for the loan is 12.5%-15%. If you wish to proceed with a loan of 4800000, please let me know, and I can provide you with further details such as the monthly EMI and total payable amount.🔍 TRACE ee674ed4
   [11:50:15.534] received: query length 76
   [11:50:15.534] pii_redaction: redacted: ['PAN']
   [11:50:17.991] agent_response: 2457ms, 564 chars

<thinking> The user is eligible for a loan, but the maximum eligible amount is 4800000, not 5000000. I should inform the user of their eligibility status and the maximum eligible amount. Additionally, I should provide the interest rate band for the loan. </thinking>

You are eligible for a loan, but the maximum eligible amount is 4800000, not 5000000. The interest rate band for the loan is 12.5%-15%. If you wish to proceed with a loan of 4800000, please let me know, and I can provide you with further details such as the monthly EMI and total payable amount.



---
# MODULE E — Deployment to Lambda + API Gateway + Frontend
### Package the agent, deploy it, expose an HTTPS API, and connect the web UI

This is the clean, consolidated deployment path. It replaces the trial-and-error cells
from the working session. Run these in order — each is idempotent (safe to re-run).

**Key fixes baked in:**
- Handler written without `textwrap.dedent` (avoids the leading-indent syntax error)
- IAM role uses the correct `service-role/AWSLambdaBasicExecutionRole` ARN
- Lambda deployed via S3 (Strands package is ~74 MB, over the 69 MB direct limit)
- CORS enabled on API Gateway so the browser frontend can call it

In [19]:
# [E-1] Create the Lambda IAM role (idempotent)
ROLE_NAME = 'loangenie-lambda-role'
iam = boto3.client('iam', region_name=REGION)

try:
    iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps({
            "Version":"2012-10-17",
            "Statement":[{"Effect":"Allow","Principal":{"Service":"lambda.amazonaws.com"},
                          "Action":"sts:AssumeRole"}]
        })
    )
    print('✅ Role created')
except iam.exceptions.EntityAlreadyExistsException:
    print('✅ Role already exists')
except Exception as e:
    if 'AccessDenied' in str(e):
        print('❌ AccessDenied creating the IAM role.')
        print('   Your SageMaker execution role needs iam:CreateRole, iam:AttachRolePolicy,')
        print('   iam:PutRolePolicy, iam:PassRole — scoped to role/loangenie-* is enough.')
        print('   Ask your AWS admin to attach that, then re-run this cell.')
    raise

for policy in [
    'arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole',
    'arn:aws:iam::aws:policy/AmazonBedrockFullAccess',
    'arn:aws:iam::aws:policy/AmazonDynamoDBFullAccess',
    'arn:aws:iam::aws:policy/AmazonS3FullAccess',
]:
    try:
        iam.attach_role_policy(RoleName=ROLE_NAME, PolicyArn=policy)
        print(f'✅ {policy.split("/")[-1]}')
    except Exception as e:
        print(f'   {policy.split("/")[-1]}: {e}')

print('Waiting 10s for IAM propagation...')
time.sleep(10)
ROLE_ARN = f'arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}'
print(f'✅ Role ready: {ROLE_ARN}')


✅ Role already exists
✅ AWSLambdaBasicExecutionRole
✅ AmazonBedrockFullAccess
✅ AmazonDynamoDBFullAccess
✅ AmazonS3FullAccess
Waiting 10s for IAM propagation...


✅ Role ready: arn:aws:iam::831963379350:role/loangenie-lambda-role


In [20]:
# [E-2] Write the Lambda handler — NO textwrap (avoids indent error)
HANDLER = """import json
import logging
from strands import Agent, tool
from strands.models import BedrockModel

logger = logging.getLogger()
logger.setLevel(logging.INFO)

@tool
def check_eligibility(monthly_income: float, credit_score: float, loan_amount: float) -> str:
    \"\"\"Check loan eligibility.\"\"\"
    max_loan = monthly_income * 60
    reasons, eligible = [], True
    if credit_score < 650: eligible=False; reasons.append("Credit score below 650")
    if loan_amount > max_loan: eligible=False; reasons.append("Exceeds max eligible")
    if monthly_income < 25000: eligible=False; reasons.append("Income below 25000")
    band = "10.5%-12.5%" if credit_score >= 750 else "12.5%-15%"
    return json.dumps({"eligible":eligible,"max_eligible_amount":int(max_loan),
                       "reasons":reasons or ["Meets all criteria"],"interest_rate_band":band})

@tool
def calculate_emi(principal: float, annual_rate: float, tenure_months: int) -> str:
    \"\"\"Calculate monthly EMI.\"\"\"
    r = annual_rate/12/100
    emi = principal/tenure_months if r==0 else principal*r*(1+r)**tenure_months/((1+r)**tenure_months-1)
    return json.dumps({"monthly_emi":round(emi,2),"total_payable":round(emi*tenure_months,2),
                       "total_interest":round(emi*tenure_months-principal,2)})

agent = Agent(
    model=BedrockModel(model_id="us.amazon.nova-pro-v1:0"),
    system_prompt="You are LoanGenie, a loan eligibility and advisory agent. Use check_eligibility to assess eligibility and calculate_emi for repayments. Format responses in clean markdown with ## headers and **bold** figures. Never promise final approval.",
    tools=[check_eligibility, calculate_emi]
)

def lambda_handler(event, context):
    try:
        body = json.loads(event.get("body", "{}"))
        msg  = body.get("message", "").strip()
        if not msg:
            return {"statusCode":400,"body":json.dumps({"error":"message required"})}
        resp = agent(msg)
        return {
            "statusCode":200,
            "headers":{
                "Content-Type":"application/json",
                "Access-Control-Allow-Origin":"*",
                "Access-Control-Allow-Headers":"Content-Type",
                "Access-Control-Allow-Methods":"POST,OPTIONS"
            },
            "body":json.dumps({"response":str(resp)})
        }
    except Exception as e:
        logger.error(str(e))
        return {"statusCode":500,"body":json.dumps({"error":str(e)})}
"""

with open('/tmp/lambda_handler.py', 'w') as f:
    f.write(HANDLER)

# Verify the file starts at column 0 (no indent on line 2)
with open('/tmp/lambda_handler.py') as f:
    lines = f.readlines()
print('First 2 lines (must have NO leading spaces):')
for i, l in enumerate(lines[:2], 1):
    print(f'  Line {i}: {repr(l)}')
print('✅ Handler written cleanly')

First 2 lines (must have NO leading spaces):
  Line 1: 'import json\n'
  Line 2: 'import logging\n'
✅ Handler written cleanly


In [21]:
# [E-3] Package the agent and upload the zip to S3
S3_BUCKET = f'loangenie-kb-{ACCOUNT_ID}-us-east-1'   # fixed: removed stray "-an" suffix that caused NoSuchBucket
S3_KEY    = 'lambda/loan_agent.zip'
pkg       = '/tmp/loan_pkg'
zip_path  = '/tmp/loan_agent.zip'

s3 = boto3.client('s3', region_name=REGION)

# Verify the bucket exists — create it if it doesn't, instead of failing at upload time
existing_buckets = [b['Name'] for b in s3.list_buckets()['Buckets']]
if S3_BUCKET not in existing_buckets:
    print(f'⚠️  Bucket {S3_BUCKET} not found. Creating it...')
    if REGION == 'us-east-1':
        s3.create_bucket(Bucket=S3_BUCKET)
    else:
        s3.create_bucket(Bucket=S3_BUCKET, CreateBucketConfiguration={'LocationConstraint': REGION})
    print(f'✅ Bucket created: {S3_BUCKET}')
else:
    print(f'✅ Bucket exists: {S3_BUCKET}')

print('Installing packages into build dir (2-3 min)...')
shutil.rmtree(pkg, ignore_errors=True); os.makedirs(pkg)
subprocess.run(f'pip install strands-agents strands-agents-tools boto3 -t {pkg} -q',
               shell=True, check=True)
shutil.copy('/tmp/lambda_handler.py', f'{pkg}/lambda_handler.py')

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(pkg):
        for file in files:
            fp = os.path.join(root, file)
            zf.write(fp, os.path.relpath(fp, pkg))
print(f'✅ Packaged: {os.path.getsize(zip_path)/1024/1024:.1f} MB')

s3.upload_file(zip_path, S3_BUCKET, S3_KEY)
print(f'✅ Uploaded: s3://{S3_BUCKET}/{S3_KEY}')


⚠️  Bucket loangenie-kb-831963379350-us-east-1 not found. Creating it...


✅ Bucket created: loangenie-kb-831963379350-us-east-1
Installing packages into build dir (2-3 min)...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
dash 2.18.1 requires dash-core-components==2.0.0, which is not installed.
dash 2.18.1 requires dash-html-components==2.0.0, which is not installed.
dash 2.18.1 requires dash-table==5.0.0, which is not installed.
skops 0.14.0 requires prettytable>=3.9, which is not installed.
aiobotocore 3.7.0 requires botocore<1.43.1,>=1.42.90, but you have botocore 1.43.51 which is incompatible.
amazon-sagemaker-sql-editor 0.2.5 requires rpds-py==0.27.1, but you have rpds-py 2026.6.3 which is incom

✅ Packaged: 74.3 MB


✅ Uploaded: s3://loangenie-kb-831963379350-us-east-1/lambda/loan_agent.zip


In [22]:
# [E-4] Deploy the Lambda function from S3
LAMBDA_NAME = 'LoanGenieAgent'
lam = boto3.client('lambda', region_name=REGION)

try:
    resp = lam.create_function(
        FunctionName=LAMBDA_NAME, Runtime='python3.12', Role=ROLE_ARN,
        Handler='lambda_handler.lambda_handler',
        Code={'S3Bucket': S3_BUCKET, 'S3Key': S3_KEY},
        MemorySize=1024, Timeout=300,
    )
    print(f'✅ Lambda created: {resp["FunctionArn"]}')
except lam.exceptions.ResourceConflictException:
    resp = lam.update_function_code(
        FunctionName=LAMBDA_NAME, S3Bucket=S3_BUCKET, S3Key=S3_KEY)
    print(f'✅ Lambda updated: {resp["FunctionArn"]}')

LAMBDA_ARN = f'arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{LAMBDA_NAME}'
print('Waiting 15s for Lambda to be ready...')
time.sleep(15)

# Verify it works before exposing an API
resp = lam.invoke(FunctionName=LAMBDA_NAME, InvocationType='RequestResponse',
    Payload=json.dumps({'body': json.dumps({'message': 'Am I eligible for 500000 with income 80000 and score 720?'})}))
raw = json.loads(resp['Payload'].read())
if raw.get('statusCode') == 200:
    print('✅ Lambda test passed:')
    show(json.loads(raw['body'])['response'])
else:
    print('❌ Lambda error:', raw)

✅ Lambda updated: arn:aws:lambda:us-east-1:831963379350:function:LoanGenieAgent
Waiting 15s for Lambda to be ready...


✅ Lambda test passed:


## Loan Eligibility Assessment

**Eligibility:** You are **eligible** for a loan.

**Maximum Eligible Amount:** You can be eligible for a loan amount up to **4800000**.

**Reasons for Eligibility:**
- Meets all criteria

**Interest Rate Band:** The interest rate for your loan could be between **12.5%-15%**.

**Note:** This assessment indicates your eligibility based on the provided information. Final loan approval is subject to additional verification and terms set by the lender.


In [23]:
# [E-5] Create the API Gateway HTTP endpoint with CORS (idempotent — reuses existing API by name)
apigw = boto3.client('apigatewayv2', region_name=REGION)

existing_apis = apigw.get_apis()['Items']
match = next((a for a in existing_apis if a['Name'] == 'LoanGenieAPI'), None)

if match:
    API_ID = match['ApiId']
    print(f'ℹ️  Reusing existing API: {API_ID}')
else:
    api = apigw.create_api(
        Name='LoanGenieAPI', ProtocolType='HTTP',
        CorsConfiguration={
            'AllowOrigins': ['*'],
            'AllowMethods': ['POST', 'OPTIONS'],
            'AllowHeaders': ['Content-Type'],
        }
    )
    API_ID = api['ApiId']
    print(f'✅ API created: {API_ID}')

    integration = apigw.create_integration(
        ApiId=API_ID, IntegrationType='AWS_PROXY',
        IntegrationUri=LAMBDA_ARN, PayloadFormatVersion='2.0')
    apigw.create_route(ApiId=API_ID, RouteKey='POST /advise',
        Target=f'integrations/{integration["IntegrationId"]}')
    apigw.create_stage(ApiId=API_ID, StageName='prod', AutoDeploy=True)

try:
    lam.add_permission(
        FunctionName=LAMBDA_NAME, StatementId='allow-apigw-loangenie',
        Action='lambda:InvokeFunction', Principal='apigateway.amazonaws.com',
        SourceArn=f'arn:aws:execute-api:{REGION}:{ACCOUNT_ID}:{API_ID}/*')
except lam.exceptions.ResourceConflictException:
    pass

API_ENDPOINT = f'https://{API_ID}.execute-api.{REGION}.amazonaws.com/prod/advise'
print('✅ API Gateway deployed!')
print(f'\n📌 PASTE THIS INTO THE FRONTEND CONNECTION BOX:')
print(f'   {API_ENDPOINT}')


ℹ️  Reusing existing API: 1a8nbql46e
✅ API Gateway deployed!

📌 PASTE THIS INTO THE FRONTEND CONNECTION BOX:
   https://1a8nbql46e.execute-api.us-east-1.amazonaws.com/prod/advise


In [32]:
# [E-6] Final end-to-end test through the live API
payload = json.dumps({'message':
    'I earn 90000 with credit score 760. Am I eligible for 700000, '
    'and what would the EMI be over 48 months at 11%?'}).encode()
req = urllib.request.Request(API_ENDPOINT, data=payload,
    headers={'Content-Type':'application/json'}, method='POST')

try:
    with urllib.request.urlopen(req) as r:
        data = json.loads(r.read())
    print('✅ Live API response:')
    show(data['response'])
except urllib.error.HTTPError as e:
    print(f'❌ HTTP {e.code}: {e.reason}')
    print(e.read().decode())
    print('\nPulling recent CloudWatch logs to find the real error...')
    logs = boto3.client('logs', region_name=REGION)
    log_group = f'/aws/lambda/{LAMBDA_NAME}'
    try:
        resp = logs.filter_log_events(
            logGroupName=log_group,
            startTime=int((time.time() - 300) * 1000),
            filterPattern='?ERROR ?Exception ?Traceback ?"Task timed out"'
        )
        for ev in resp['events']:
            print(ev['message'])
    except Exception as log_err:
        print(f'   Could not fetch logs: {log_err}')


✅ Live API response:


## Loan Eligibility and EMI Calculation

### Loan Eligibility
- **Eligibility Status:** Eligible
- **Maximum Eligible Amount:** **5,400,000**
- **Reasons for Eligibility:** Meets all criteria
- **Interest Rate Band:** **10.5%-12.5%**

### EMI Calculation for 700,000 over 48 months at 11%
- **Monthly EMI:** **18,091.87**
- **Total Payable Amount:** **868,409.56**
- **Total Interest Payable:** **168,409.56**

**Note:** This is an indicative calculation. Final approval and terms may vary.


---
# 🏆 Capstone — Production LoanGenie
### Every advanced capability combined into one system

Supervisor routing + PII redaction + cost tracking + tracing, all in one call path.

In [33]:
def production_loangenie(user_query, applicant_id='demo'):
    """Full production pipeline with all advanced capabilities."""
    trace_id = str(uuid.uuid4())[:8]
    start = time.time()

    # 1. PII redaction
    clean_query, pii_found = redact_pii(user_query)

    # 2. Supervisor routing to specialists
    response = str(supervisor(clean_query))

    # 3. Metrics (approximate token counts for demo)
    latency_ms = (time.time() - start) * 1000
    metrics.track(clean_query, response,
                  len(clean_query.split())*2, len(response.split())*2, latency_ms)

    # 4. Structured result
    return {
        'trace_id': trace_id,
        'answer': response,
        'pii_redacted': pii_found,
        'latency_ms': round(latency_ms, 1),
        'applicant_id': applicant_id
    }


# Full end-to-end test
result = production_loangenie(
    'My PAN is ABCDE1234F. I earn 90000 with a credit score of 760. '
    'Am I eligible for 600000, what would the EMI be over 48 months at 12%, '
    'and what documents do I need?',
    applicant_id='applicant-priya-002'
)

print(f'🔍 Trace: {result["trace_id"]}')
print(f'🔒 PII redacted: {result["pii_redacted"]}')
print(f'⏱  Latency: {result["latency_ms"]}ms')
show(result["answer"], title="LoanGenie Response")
print(f'\n📊 Session cost so far: ${metrics.summary()["total_cost_usd"]}')

<thinking> The user's query includes questions about loan eligibility, EMI calculations, and required documents. Therefore,

 I need to call all three specialists: eligibility_specialist, advisory_specialist, and documents_specialist

. </thinking> 
Tool #7: eligibility_specialist

Tool #8: advisory_specialist

Tool #9: documents_specialist


<thinking><thinking> I need to check the eligibility of the user for a loan amount of 60000

 I need to calculate the EMI for a loan of 600000 INR over0 given their monthly income of 90000 and credit score of 760

...

,800.30**
- **Total Payable:** **₹758,414.46**
- **Total Interest:** **₹158,41

4.46**

## Required Documents
- PAN Card
- Aadhaar Card
- Income Tax Returns (ITR) for the last 2 years
- Salary Slips for the last

 3 months
- Bank Statements for the last 6 months
- Employment Letter or Appointment Letter
- Proof of Address (Utility Bill or Rental Agreement)🔍 Trace: dcafa255
🔒 PII redacted: ['PAN']
⏱  Latency: 9474.9ms


### LoanGenie Response

## Loan Eligibility
- **Eligible:** Yes
- **Max Eligible Amount:** ₹54,00,000
- **Reasons:** Meets all criteria
- **Interest Rate Band:** 10.5%-12.5%

## EMI Calculation
- **Loan Amount:** ₹6,00,000
- **Interest Rate:** 12%
- **Tenure:** 48 months
- **Monthly EMI:** **₹15,800.30**
- **Total Payable:** **₹758,414.46**
- **Total Interest:** **₹158,414.46**

## Required Documents
- PAN Card
- Aadhaar Card
- Income Tax Returns (ITR) for the last 2 years
- Salary Slips for the last 3 months
- Bank Statements for the last 6 months
- Employment Letter or Appointment Letter
- Proof of Address (Utility Bill or Rental Agreement)



📊 Session cost so far: $0.004544


---
# 🧹 Cleanup Advanced Resources
Removes only the resources created in this notebook (guardrail, CloudWatch metrics).

In [35]:
def cleanup_advanced():
    bedrock_client = boto3.client('bedrock', region_name=REGION)
    # Delete guardrail
    try:
        gids = bedrock_client.list_guardrails()['guardrails']
        for g in gids:
            if g['name'] == 'loangenie-guardrail':
                bedrock_client.delete_guardrail(guardrailIdentifier=g['id'])
                print(f'✅ Deleted guardrail: {g["id"]}')
    except Exception as e:
        print(f'Guardrail: {e}')
    print('✅ CloudWatch metrics expire automatically (no deletion needed)')
    print('\nNote: core resources (KB, DynamoDB, AgentCore) are shared with Part 1 —')
    print('      delete them via Part 1 cleanup when fully done.')
cleanup_advanced()
print('Uncomment cleanup_advanced() to run')

✅ Deleted guardrail: ygqbclvqr9wj
✅ CloudWatch metrics expire automatically (no deletion needed)

Note: core resources (KB, DynamoDB, AgentCore) are shared with Part 1 —
      delete them via Part 1 cleanup when fully done.
Uncomment cleanup_advanced() to run


---
## ✅ Summary — Advanced LoanGenie

You extended LoanGenie from a working agent into a **production-grade system**:

| Module | Capability | Production value |
|--------|-----------|------------------|
| A | Multi-agent orchestration | Specialist agents, easier to tune & debug |
| B | Streaming + async tools | Real-time UX, lower latency |
| C | Guardrails + PII redaction | Regulatory compliance, data protection |
| D | Observability | Cost control, performance monitoring, tracing |

**What makes this production-ready vs a demo:**
- Every customer input is PII-redacted before it touches the model or logs
- Denied topics (guaranteed approval, tax evasion) are blocked by guardrails
- Every query is cost-tracked and traced — you know exactly what you're spending
- The supervisor pattern means each specialist can be tested and improved independently

---
**Share your build on LinkedIn — tag K21 Academy and Atul Kumar!**
Community: https://www.skool.com/k21academy